# Week 1 — PyTorch basics: solubility regression on ESOL

Goal: build the full training pipeline by hand — no Lightning, no wandb. See `README.md` for the full brief and hard constraints.

Pipeline: SMILES -> Morgan fingerprint -> `Dataset` -> `DataLoader` -> `nn.Module` -> training loop -> validation loop -> checkpoint.

Each section below has a markdown cell describing the step and a code cell with a stub. Fill in the `TODO`s yourself — don't skip to the answer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

from rdkit import Chem
from rdkit.Chem import AllChem

from datasets import load_dataset

# TODO: pick a random seed and set it for numpy + torch, so your run is reproducible.
SEED = None

## 1. Load the ESOL dataset

Search the HuggingFace Hub (https://huggingface.co/datasets) for an ESOL / MoleculeNet aqueous-solubility dataset. Don't assume a repo id — search for it and check the dataset card so you know what the columns actually are before you write code against them.

Once loaded, inspect it: how many rows, what are the column names, which column is the SMILES string, which column is the solubility target (and what units / scale is it on — raw or log solubility)?

In [ ]:
# TODO: dataset = load_dataset("<repo_id you found>")
dataset = None

# TODO: inspect it - print the number of rows, the column names, and a couple of example rows.


## 2. Featurize: SMILES -> Morgan fingerprint

Write a function that turns one SMILES string into a fixed-length numeric vector using RDKit's Morgan fingerprint. You'll need to decide (and be able to justify) a `radius` and a bit-vector length (`n_bits`).

Then apply it across the whole dataset to build your feature matrix `X` and target vector `y`. Watch for SMILES strings that fail to parse — decide what you do with them (drop, or investigate why).

In [ ]:
def smiles_to_fingerprint(smiles: str, radius: int = None, n_bits: int = None) -> np.ndarray:
    """Convert one SMILES string to a Morgan fingerprint bit vector."""
    # TODO: Chem.MolFromSmiles, then compute the fingerprint.
    # TODO: what does MolFromSmiles return if the SMILES is invalid? Handle that case.
    # TODO: return a numpy array, not an RDKit bit-vector object.
    raise NotImplementedError


# TODO: apply smiles_to_fingerprint to every molecule, build X (N x n_bits) and y (N,) arrays.
X = None
y = None

## 3. Train / validation split

Split `X`/`y` into a training set and a validation set. Think about the split fraction and whether you need to set a random state for reproducibility.

In [ ]:
# TODO: X_train, X_val, y_train, y_val = train_test_split(...)
X_train = X_val = y_train = y_val = None

## 4. `Dataset` and `DataLoader`

Write a custom `torch.utils.data.Dataset` that wraps your feature/target arrays. It needs `__len__` and `__getitem__`. Think about dtypes: what dtype does `nn.Linear` expect, and what dtype are your fingerprint bits / targets in right now?

In [ ]:
class ESOLDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # TODO: store X and y as tensors with the right dtype.
        raise NotImplementedError

    def __len__(self):
        # TODO
        raise NotImplementedError

    def __getitem__(self, idx):
        # TODO: return (features, target) for this index.
        raise NotImplementedError


# TODO: instantiate train_dataset / val_dataset
train_dataset = None
val_dataset = None

# TODO: wrap both in DataLoader. Pick a batch size. Should train and val use the same shuffle setting?
BATCH_SIZE = None
train_loader = None
val_loader = None

## 5. Model — `nn.Module`

A small MLP: input layer sized to your fingerprint length, two or three hidden layers, and a single output for the regression target. Think about what (if any) activation belongs on the output layer for a regression task, versus a classification one.

In [ ]:
class SolubilityRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = None):
        super().__init__()
        # TODO: define your layers (nn.Linear, activation, ...).
        raise NotImplementedError

    def forward(self, x):
        # TODO: wire the layers together and return the prediction.
        raise NotImplementedError


# TODO: instantiate the model with the right input_dim.
model = None

## 6. Training loop

Pick a loss function appropriate for regression, and an optimizer. Write the loop: for each epoch, iterate over `train_loader`, do the forward pass, compute loss, zero gradients, backward, step. Track the average training loss per epoch so you can plot it later.

In [ ]:
# TODO: pick a loss function for regression.
loss_fn = None

# TODO: pick an optimizer and learning rate.
optimizer = None

N_EPOCHS = None
train_losses = []

def train_one_epoch(model, loader, loss_fn, optimizer) -> float:
    """Run one training epoch, return the average training loss."""
    # TODO: model.train(), loop over batches, forward/backward/step, accumulate loss.
    raise NotImplementedError

# TODO: loop for N_EPOCHS, call train_one_epoch, append to train_losses, print progress.


## 7. Validation loop

Write a separate function that evaluates the model on `val_loader` without updating weights. What does the model need to be set to, and what context manager avoids wasting memory on gradients you won't use? Report validation loss plus at least one interpretable metric (RMSE, MAE, or R^2 - `sklearn.metrics` has these) alongside the raw loss.

In [ ]:
def evaluate(model, loader, loss_fn) -> dict:
    """Run validation, return a dict of metrics (loss, mae, r2, ...)."""
    # TODO: model.eval(), no_grad, loop over batches, collect predictions and targets.
    # TODO: compute loss and at least one of MAE / RMSE / R2 over the full val set.
    raise NotImplementedError


val_losses = []
# TODO: fold evaluate() into your training loop above (once per epoch), or re-run it here
# against the final model - your choice, but decide deliberately and note why in REFLECTION.md.

## 8. Checkpointing

Save the model's weights to disk (not the whole model object - just the `state_dict`, plus whatever else you'd need to resume: optimizer state, epoch number, best val loss). Then load it back into a fresh model instance and confirm it reproduces the same validation metric you got above.

In [ ]:
CHECKPOINT_PATH = "checkpoint.pt"

# TODO: save a checkpoint - what needs to be in the dict you save?

# TODO: load it back into a *new* model instance and re-run evaluate() to confirm the
# metrics match what you saw before saving.


## 9. Look at what you built

Plot train (and val, if you tracked it per-epoch) loss curves over epochs. Does it look like it's still improving, plateaued, or overfitting? Take this - and anything that broke along the way - into `REFLECTION.md`.

In [ ]:
# TODO: plot train_losses (and val_losses if you have them) vs epoch.
